# BoltzGen Nanobody Scoring
**Target:** P05231 (IL-6)  
**Source:** [github.com/schneider-weave/boltz_scoring](https://github.com/schneider-weave/boltz_scoring)

This notebook:
1. Clones the repo (scoring inputs + boltzgen source)
2. Installs boltzgen and dependencies
3. Runs boltzgen scoring on all 227 nanobody sequences
4. Displays and saves ranked results

> **Kaggle settings:** Enable GPU (P100 or T4), enable Internet access.

## 1. Clone repo & install boltzgen

In [ ]:
import os

REPO_URL  = "https://github.com/schneider-weave/boltz_scoring"
REPO_DIR  = "/kaggle/working/boltz_scoring"
CACHE_DIR = "/kaggle/working/cache"   # model weights (~6 GB)

# Clone repo
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

In [ ]:
%%bash
# Install CUDA-compatible torch first (Kaggle GPU images are CUDA 12.x)
pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# Install cuequivariance wheels (required by boltzgen, not on PyPI for all platforms)
pip install -q \
    cuequivariance-ops-torch-cu12 \
    cuequivariance-torch \
    cuequivariance-ops-cu12

# Install boltzgen from the cloned source
pip install -q -e /kaggle/working/boltz_scoring/boltzgen

echo "Installation complete"

In [ ]:
# Verify GPU and boltzgen CLI are available
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!boltzgen --version 2>/dev/null || boltzgen --help 2>&1 | head -3

## 2. Configuration

In [ ]:
import os
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_DIR       = Path("/kaggle/working/boltz_scoring")
SCORING_INPUTS = REPO_DIR / "scoring_inputs"        # 227 per-nanobody YAML files
MSA_DIR        = REPO_DIR / "msa_files"              # P05231.a3m
OUTPUT_DIR     = Path("/kaggle/working/scoring_results")
CACHE_DIR      = Path("/kaggle/working/cache")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Scoring settings ──────────────────────────────────────────────────────────
PROTOCOL    = "nanobody-anything"
NUM_DESIGNS = 1      # 1 fold per nanobody (scoring mode)

# Count input files
yaml_files = list(SCORING_INPUTS.glob("*.yaml"))
print(f"Input YAMLs  : {len(yaml_files)}")
print(f"MSA file     : {MSA_DIR / 'P05231.a3m'}  exists={( MSA_DIR / 'P05231.a3m').exists()}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Cache dir    : {CACHE_DIR}")
print(f"Protocol     : {PROTOCOL}")

## 3. (Optional) Preview one input YAML

In [ ]:
import yaml

sample = yaml_files[0]
print(f"File: {sample.name}\n")
with open(sample) as f:
    print(f.read())

## 4. Download model weights
Downloads ~6 GB to `CACHE_DIR` on first run. Subsequent runs reuse the cache.

In [ ]:
!boltzgen download all --cache {CACHE_DIR}

## 5. Run boltzgen scoring

Pipeline for scoring fixed sequences:  
`folding → analysis → filtering`  
(inverse folding is skipped — sequences are already specified)

In [ ]:
import subprocess, sys

cmd = [
    "boltzgen", "run", str(SCORING_INPUTS),
    "--output",      str(OUTPUT_DIR),
    "--protocol",    PROTOCOL,
    "--cache",       str(CACHE_DIR),
    "--skip_inverse_folding",
    "--num_designs", str(NUM_DESIGNS),
    "--no_subprocess",   # run steps in-process (better for notebooks)
]

print("Running:", " ".join(cmd))
print("-" * 60)

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode != 0:
    print("\n[ERROR] boltzgen exited with code", result.returncode)
else:
    print("\n[OK] Scoring complete")

## 6. Load & display results

In [ ]:
import pandas as pd
from pathlib import Path
import glob

# boltzgen writes metrics here after the analysis step
metrics_csv = OUTPUT_DIR / "intermediate_designs" / "aggregate_metrics_analyze.csv"

# Also check inverse_folded path (written when skip_inverse_folding is NOT used)
if not metrics_csv.exists():
    metrics_csv = OUTPUT_DIR / "intermediate_designs_inverse_folded" / "aggregate_metrics_analyze.csv"

if not metrics_csv.exists():
    # Search recursively
    found = list(OUTPUT_DIR.rglob("aggregate_metrics_analyze.csv"))
    if found:
        metrics_csv = found[0]
        print(f"Found metrics at: {metrics_csv}")
    else:
        raise FileNotFoundError(
            f"aggregate_metrics_analyze.csv not found under {OUTPUT_DIR}\n"
            f"Contents: {list(OUTPUT_DIR.rglob('*'))[:20]}"
        )

df = pd.read_csv(metrics_csv)
print(f"Loaded {len(df)} results, {len(df.columns)} columns")
df.head()

In [ ]:
# Key scoring metrics — keep only what's available
DESIRED_METRICS = [
    "id",
    "delta_sasa_refolded",        # buried surface area  (higher = better binding)
    "plip_hbonds_refolded",       # H-bonds at interface (higher = better)
    "largest_hydrophobic_refolded",# hydrophobic patch    (lower = better)
    "noncovalents_refolded",      # non-covalent contacts
    "plddt",                      # folding confidence   (higher = better)
    "ptm",                        # predicted TM-score   (higher = better)
    "iptm",                       # interface pTM        (higher = better)
]

available = [c for c in DESIRED_METRICS if c in df.columns]
missing   = [c for c in DESIRED_METRICS if c not in df.columns]
if missing:
    print(f"Note: columns not in output: {missing}")

df_scores = df[available].copy()

# Parse nanobody ID from the 'id' column  (format: design_spec_XXXX_rank=N_P05231)
df_scores["nanobody_id"] = df_scores["id"].str.extract(r"(design_spec_\d+)", expand=False)
df_scores["original_rank"] = df_scores["id"].str.extract(r"rank=(\d+)", expand=False).astype(float)

df_scores.head()

In [ ]:
# Rank by delta_sasa_refolded (primary binding indicator), then h-bonds
sort_cols = [c for c in ["delta_sasa_refolded", "plip_hbonds_refolded", "iptm"] if c in df_scores.columns]

df_ranked = df_scores.sort_values(sort_cols, ascending=False).reset_index(drop=True)
df_ranked.index += 1  # rank starts at 1
df_ranked.index.name = "boltzgen_rank"

print(f"Top 20 nanobodies (ranked by {sort_cols}):")
df_ranked.head(20)

## 7. Visualize score distributions

In [ ]:
import matplotlib.pyplot as plt

plot_metrics = [c for c in [
    "delta_sasa_refolded",
    "plip_hbonds_refolded",
    "largest_hydrophobic_refolded",
    "plddt",
    "iptm",
] if c in df_ranked.columns]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(4 * len(plot_metrics), 4))
if len(plot_metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, plot_metrics):
    ax.hist(df_ranked[metric].dropna(), bins=30, edgecolor="white", color="steelblue")
    ax.set_title(metric.replace("_", "\n"), fontsize=9)
    ax.set_xlabel("score")
    ax.set_ylabel("count")

plt.suptitle("BoltzGen Score Distributions — P05231 Nanobodies", fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: score_distributions.png")

In [ ]:
# Scatter: delta_sasa vs h-bonds (binding quality overview)
x_col = "delta_sasa_refolded"
y_col = "plip_hbonds_refolded"

if x_col in df_ranked.columns and y_col in df_ranked.columns:
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(
        df_ranked[x_col], df_ranked[y_col],
        c=df_ranked.index, cmap="viridis_r", alpha=0.8, s=40
    )
    plt.colorbar(sc, ax=ax, label="boltzgen_rank")
    ax.set_xlabel("delta_sasa_refolded  (higher = better)")
    ax.set_ylabel("plip_hbonds_refolded  (higher = better)")
    ax.set_title("Binding Quality — P05231 Nanobodies")

    # Label top 5
    for i, row in df_ranked.head(5).iterrows():
        ax.annotate(str(row.get("nanobody_id", i)),
                    (row[x_col], row[y_col]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")

    plt.tight_layout()
    plt.savefig("/kaggle/working/sasa_vs_hbonds.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: sasa_vs_hbonds.png")

## 8. Save results

In [ ]:
# Save full ranked table
out_csv = "/kaggle/working/nanobody_boltzgen_scores.csv"
df_ranked.to_csv(out_csv)
print(f"Saved: {out_csv}")

# Summary stats
print("\nSummary statistics:")
df_ranked[plot_metrics].describe().round(3)

In [ ]:
# Print top 10 with sequences
# Load sequences back from the input YAMLs to attach to results
import yaml as pyyaml
from pathlib import Path

seq_map = {}
for yf in Path(SCORING_INPUTS).glob("*.yaml"):
    with open(yf) as f:
        data = pyyaml.safe_load(f)
    nb_id = yf.stem  # e.g. design_spec_0673_rank=4
    for entity in data.get("entities", []):
        if entity.get("protein", {}).get("id") == "B":
            seq_map[nb_id] = entity["protein"]["sequence"]

df_ranked["sequence"] = df_ranked["nanobody_id"].map(
    lambda x: next((v for k, v in seq_map.items() if x and x in k), None)
)

top10_cols = ["nanobody_id", "original_rank"] + sort_cols + ["sequence"]
top10_cols = [c for c in top10_cols if c in df_ranked.columns]

print("Top 10 nanobodies by BoltzGen scoring:")
df_ranked[top10_cols].head(10)

In [ ]:
# Save top 10 with sequences as FASTA
fasta_out = "/kaggle/working/top10_boltzgen_scored.fasta"
with open(fasta_out, "w") as f:
    for rank, row in df_ranked.head(10).iterrows():
        nb_id = row.get("nanobody_id", f"nanobody_{rank}")
        sasa  = row.get("delta_sasa_refolded", "NA")
        hb    = row.get("plip_hbonds_refolded", "NA")
        seq   = row.get("sequence", "")
        f.write(f">{nb_id}|boltzgen_rank={rank}|delta_sasa={sasa:.2f}|hbonds={hb}\n{seq}\n")

print(f"Saved: {fasta_out}")
!cat {fasta_out}

## Output files

| File | Description |
|---|---|
| `nanobody_boltzgen_scores.csv` | Full ranked table with all metrics |
| `top10_boltzgen_scored.fasta` | Top 10 sequences with scores in header |
| `score_distributions.png` | Histogram of each scoring metric |
| `sasa_vs_hbonds.png` | Scatter plot: binding surface vs H-bonds |
| `scoring_results/` | Full boltzgen output (CIF structures, NPZ files) |

## Key metrics explained

| Metric | Direction | Meaning |
|---|---|---|
| `delta_sasa_refolded` | ↑ higher = better | Buried surface area — how much surface is hidden at the interface |
| `plip_hbonds_refolded` | ↑ higher = better | Number of H-bonds between nanobody and target |
| `largest_hydrophobic_refolded` | ↓ lower = better | Size of hydrophobic patch (developability concern) |
| `plddt` | ↑ higher = better | Per-residue confidence from the folding model |
| `iptm` | ↑ higher = better | Interface predicted TM-score (complex quality) |